In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gamma

# =========================
# Target Distribution
# =========================

def target_distribution(x):
    """
    Target distribution:
        pi(x) ∝ exp(-x^4)

    We only need the unnormalized density for Metropolis.
    """
    return np.exp(-x**4)


# =========================
# Proposal Distribution PDFs
# =========================

def distribution_1(y, x, step_size):
    """
    Uniform proposal PDF:
        q(y | x) = Uniform(x - step_size, x + step_size)
    """
    if x - step_size <= y <= x + step_size:
        return 1 / (2 * step_size)
    return 0


def distribution_2(y, x, step_size):
    """
    Normal proposal PDF:
        q(y | x) = Normal(x, step_size^2)
    """
    return (1 / (np.sqrt(2 * np.pi) * step_size)) * np.exp(
        -((y - x) ** 2) / (2 * step_size**2)
    )


def distribution_3(y, x, step_size):
    """
    Laplace proposal PDF:
        q(y | x) = Laplace(x, step_size)
    """
    return (1 / (2 * step_size)) * np.exp(
        -abs(y - x) / step_size
    )


# =========================
# Proposal Samplers
# =========================

def sample_distribution_1(x, step_size):
    """
    Sample from Uniform(x - step_size, x + step_size)
    """
    return np.random.uniform(x - step_size, x + step_size)


def sample_distribution_2(x, step_size):
    """
    Sample from Normal(x, step_size^2)

    This does not use np.random.normal.
    It uses the Box-Muller transform.
    """
    u1 = np.random.uniform(0, 1)
    u2 = np.random.uniform(0, 1)

    z = np.sqrt(-2 * np.log(u1)) * np.cos(2 * np.pi * u2)

    return x + step_size * z


def sample_distribution_3(x, step_size):
    """
    Sample from Laplace(x, step_size)

    This does not use np.random.laplace.
    It uses inverse transform sampling.
    """
    u = np.random.uniform(-0.5, 0.5)

    epsilon = -step_size * np.sign(u) * np.log(1 - 2 * abs(u))

    return x + epsilon


# =========================
# One Metropolis-Hastings Step
# =========================

def metropolis_step(x, step_size, proposal_pdf, proposal_sampler):
    """
    One Metropolis-Hastings update.

    x: current state
    step_size: proposal scale
    proposal_pdf: distribution_1, distribution_2, or distribution_3
    proposal_sampler: sample_distribution_1, sample_distribution_2, or sample_distribution_3
    """

    proposed_x = proposal_sampler(x, step_size)

    numerator = target_distribution(proposed_x) * proposal_pdf(x, proposed_x, step_size)
    denominator = target_distribution(x) * proposal_pdf(proposed_x, x, step_size)

    acceptance_probability = min(1, numerator / denominator)

    u = np.random.uniform(0, 1)

    if u <= acceptance_probability:
        return proposed_x, True
    else:
        return x, False


# =========================
# Metropolis-Hastings Sampler
# =========================

def metropolis_sampler(
    initial_value=0,
    n=1000,
    step_size=1.0,
    proposal_pdf=distribution_1,
    proposal_sampler=sample_distribution_1,
    burnin=0,
    lag=1
):
    """
    Runs the Metropolis-Hastings sampler.

    initial_value: starting value of the Markov chain
    n: number of samples to keep
    step_size: proposal scale
    proposal_pdf: proposal density q(y | x)
    proposal_sampler: function that samples candidate y
    burnin: number of early samples to discard
    lag: number of MH steps between kept samples
    """

    samples = np.zeros(n)
    accepted_record = np.zeros(n, dtype=bool)

    current_state = initial_value

    # Burn-in period
    for i in range(burnin):
        current_state, accepted = metropolis_step(
            current_state,
            step_size,
            proposal_pdf,
            proposal_sampler
        )

    # Sampling period
    for i in range(n):
        for j in range(lag):
            current_state, accepted = metropolis_step(
                current_state,
                step_size,
                proposal_pdf,
                proposal_sampler
            )

        samples[i] = current_state
        accepted_record[i] = accepted

    acceptance_rate = np.mean(accepted_record)

    return samples, accepted_record, acceptance_rate


# =========================
# Run the Algorithm
# =========================

np.random.seed(42)

n = 1000
initial_value = 0
burnin = 100
lag = 1

samples_1, accepted_1, rate_1 = metropolis_sampler(
    initial_value=initial_value,
    n=n,
    step_size=1.0,
    proposal_pdf=distribution_1,
    proposal_sampler=sample_distribution_1,
    burnin=burnin,
    lag=lag
)

samples_2, accepted_2, rate_2 = metropolis_sampler(
    initial_value=initial_value,
    n=n,
    step_size=1.0,
    proposal_pdf=distribution_2,
    proposal_sampler=sample_distribution_2,
    burnin=burnin,
    lag=lag
)

samples_3, accepted_3, rate_3 = metropolis_sampler(
    initial_value=initial_value,
    n=n,
    step_size=0.7,
    proposal_pdf=distribution_3,
    proposal_sampler=sample_distribution_3,
    burnin=burnin,
    lag=lag
)

print("Acceptance rate for distribution_1:", rate_1)
print("Acceptance rate for distribution_2:", rate_2)
print("Acceptance rate for distribution_3:", rate_3)


# =========================
# Normalization Factor
# =========================

Z = 0.5 * gamma(1 / 4)

print("Normalization factor Z:", Z)


# =========================
# Plot Results
# =========================

x = np.linspace(-3, 3, 1000)
target = np.exp(-x**4) / Z

plt.figure(figsize=(8, 5))

plt.hist(samples_1, bins=30, density=True, alpha=0.45, label="distribution_1: Uniform")
plt.hist(samples_2, bins=30, density=True, alpha=0.45, label="distribution_2: Normal")
plt.hist(samples_3, bins=30, density=True, alpha=0.45, label="distribution_3: Laplace")

plt.plot(x, target, linewidth=2, label="Target Distribution")

plt.title("Metropolis-Hastings Sampling for f(x) = exp(-x^4) / Z")
plt.xlabel("x")
plt.ylabel("Density")
plt.legend()
plt.grid(True)

plt.show()